In [1]:
import pandas as pd
import sqlite3
from scipy.stats import shapiro, levene, mannwhitneyu

#Imports the tests that we will be using: 
#Shapiro-Wilk tests for normality
#Levene's Test tests for equal variance in both groups
#Mann-Whitney U test is a non-parametric test if the two groups fail shapiro's test

import warnings
warnings.filterwarnings('ignore')

In [2]:
csv_filename = 'ab_data.csv' 
df = pd.read_csv(csv_filename)

conn = sqlite3.connect('ecommerce.db')

df.to_sql('ab_data', conn, if_exists='replace', index=False)

294480

In [3]:
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294480 entries, 0 to 294479
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294480 non-null  int64 
 1   timestamp     294480 non-null  object
 2   group         294480 non-null  object
 3   landing_page  294480 non-null  object
 4   converted     294480 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [5]:
df.apply(lambda x: x.nunique())
#Counts the unique values inside every single column of the dataset

user_id         290585
timestamp        35993
group                2
landing_page         2
converted            2
dtype: int64

In [6]:
df.isnull().sum()
#Counts how many values are null in each column

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

In [7]:
print(df.shape)
df = df.drop_duplicates(subset = 'user_id', keep = False)
print(df.shape)

#only keeps the original user_id row and drops the rest

(294480, 5)
(286690, 5)


In [8]:
df.groupby(['group', 'landing_page']).size()

# Counts how many users in the control and treatment groups saw the old vs. new landing page.

group      landing_page
control    old_page        143293
treatment  new_page        143397
dtype: int64

In [9]:
df.groupby(['group','landing_page']).agg({'converted': 'mean'})

#shows mean conversion rate



,,converted
group,landing_page,
control,old_page,0.120173
treatment,new_page,0.118726


In [10]:
df.groupby(['group','landing_page']).agg({'converted': lambda x: x.mean() * 100})

#Shows percentage converted

,,converted
group,landing_page,
control,old_page,12.017335
treatment,new_page,11.872633


In [ ]:
pd.DataFrame(df.loc[:,'landing_page'].value_counts(normalize = True) * 100)

#value_counts counts number of times each page appears
#normalize = True turns this into a proportion e.g. 0.50018138
#pd.DataFrame() turns this series list into a 2D table

,proportion
landing_page,
new_page,50.018138
old_page,49.981862


In [14]:
df[((df['group'] == 'control') & (df['landing_page'] == 'new_page')) |((df['group'] == 'treatment') & (df['landing_page'] == 'old_page')) ]

#checks to see whether any in the control group see the new_page or vice versa

,user_id,timestamp,group,landing_page,converted


In [16]:
test_stat, pvalue = shapiro(df.loc[df["landing_page"] == "old_page", "converted"])

print("p-value:", pvalue)
print("test_stat:", test_stat)

p-value: 9.036752175764071e-178
test_stat: 0.37923878278163137


In [ ]:
test_stat, pvalue = shapiro(df.loc[df["landing_page"] == "new_page", "converted"])

print("p-value:", pvalue)
print("test_stat:", test_stat)

#we see that the pvalues are < 0.05 and so our assumption of normality is not provided. We will use non-parametric tests

p-value: 6.263942656903057e-178
test_stat: 0.37673520650313497


In [ ]:
#Now want to test with mannwhitneyu test
#H_0: There is not statistically significant difference between the old page and new page
#H_1 There is statistically significant difference between the old page and new page


test_stat, pvalue = mannwhitneyu(df.loc[df["landing_page"] == "new_page", "converted"], df.loc[df["landing_page"] == "old_page", "converted"])

print('Test Stat = %.4f, p-value = %.4f' % (test_stat, pvalue))

#We see that the pvalue >0.05 and so we fail to reject 0. There is no statistically significant difference between teh new page and the old page, so it does not bring a profit

Test Stat = 10259026653.0000, p-value = 0.2323
